# OncoLake — Notebook de faisabilité

**Objectif :** valider, *avant* de construire le moindre échafaudage, que le pont
**UniProt → AlphaFold** fonctionne en vrai sur quelques protéines, et qu'on peut
en extraire des features exploitables pour un modèle.

On teste sur 3 protéines connues, avec les deux labels représentés :

| Gène | Accession | Label |
|------|-----------|-------|
| TP53 | P04637 | suppresseur de tumeur |
| KRAS | P01116 | oncogène (proto-oncogène) |
| RB1  | P06400 | suppresseur de tumeur |

Si ce notebook tourne de bout en bout, le projet est faisable et on peut passer
à l'étape 2 (squelette du repo).

> ⚠️ **Note importante** : le README mentionnait l'URL
> `…/files/AF-{accession}-F1-model_v4.cif`. AlphaFold DB est passé à la **v6**
> (synchronisée avec UniProt 2025_03). Plutôt que de coder en dur un numéro de
> version qui changera encore, on interroge **l'API AlphaFold** qui renvoie l'URL
> du fichier *courant*. Le code reste valide à travers les futures versions.

## 0. Dépendances

```bash
uv pip install requests gemmi numpy pandas
```

- `requests` : appels HTTP vers UniProt et AlphaFold
- `gemmi` : parsing mmCIF (robuste, rapide) — on y lit le pLDDT et les coordonnées
- `numpy` / `pandas` : features et table finale

In [1]:
import time
import requests
import numpy as np
import pandas as pd
import gemmi  # parsing mmCIF

UNIPROT_BASE = "https://rest.uniprot.org/uniprotkb"
AFDB_API     = "https://alphafold.ebi.ac.uk/api/prediction"  # API, pas l'URL fichier en dur

# Les 3 protéines de test : accession -> (gène, label)
TARGETS = {
    "P04637": ("TP53", "tumor_suppressor"),
    "P01116": ("KRAS", "oncogene"),
    "P06400": ("RB1",  "tumor_suppressor"),
}

session = requests.Session()
session.headers.update({"User-Agent": "OncoLake-feasibility/0.1"})

## 1. UniProt — le traducteur gène ↔ protéine

Pour chaque accession, UniProt nous donne en une requête : le **nom de gène**, la
**séquence** (suite d'acides aminés) et les **mots-clés** (qui contiennent le label
oncogène / suppresseur). On récupère le JSON de l'entrée directement par accession.

In [6]:
def fetch_uniprot_entry(accession: str) -> dict:
    '''Récupère gène, séquence et keywords pour une accession UniProt.'''
    url = f"{UNIPROT_BASE}/{accession}.json"
    r = session.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()

    gene = None
    if data.get("genes"):
        gene = data["genes"][0].get("geneName", {}).get("value")

    sequence = data.get("sequence", {}).get("value")
    keywords = [k.get("name") for k in data.get("keywords", [])]

    return {
        "accession": accession,
        "gene": gene,
        "sequence": sequence,
        "length": len(sequence) if sequence else None,
        "keywords": keywords,
    }

# Test sur les 3 cibles
for acc in TARGETS:
    entry = fetch_uniprot_entry(acc)
    has_label = any("oncogene" in k.lower() or "suppressor" in k.lower()
                    for k in entry["keywords"])
    print(f"{entry['gene']:6s} ({acc}) | longueur={entry['length']:>4} | "
          f"label keyword présent={has_label}")
    time.sleep(0.2)  # courtoisie envers l'API

TP53   (P04637) | longueur= 393 | label keyword présent=True
KRAS   (P01116) | longueur= 189 | label keyword présent=True
RB1    (P06400) | longueur= 928 | label keyword présent=True


### 1b. Source de labels en masse (option B du README)

Au lieu de coder en dur une liste de gènes, on interroge UniProt par **mot-clé** :
`KW-0656` (Proto-oncogene) et `KW-0043` (Tumor suppressor), restreint à l'humain et
aux entrées revues (Swiss-Prot). Une seule requête renvoie accession + gène +
séquence + label. C'est ainsi qu'on remplira la zone *raw* à l'étape d'ingestion.

On limite ici à 5 résultats juste pour prouver que la requête fonctionne.

In [7]:
def search_uniprot_by_keyword(keyword_id: str, label: str, limit: int = 5) -> pd.DataFrame:
    '''Liste des protéines humaines revues portant un mot-clé donné.'''
    query = f"(keyword:{keyword_id}) AND (organism_id:9606) AND (reviewed:true)"
    params = {
        "query": query,
        "format": "json",
        "fields": "accession,gene_primary,length",
        "size": limit,
    }
    r = session.get(f"{UNIPROT_BASE}/search", params=params, timeout=30)
    r.raise_for_status()
    rows = []
    for item in r.json().get("results", []):
        gene = (item.get("genes") or [{}])[0].get("geneName", {}).get("value")
        rows.append({
            "accession": item["primaryAccession"],
            "gene": gene,
            "length": item.get("sequence", {}).get("length"),
            "label": label,
        })
    return pd.DataFrame(rows)

onco   = search_uniprot_by_keyword("KW-0656", "oncogene", limit=5)
suppr  = search_uniprot_by_keyword("KW-0043", "tumor_suppressor", limit=5)
print("Échantillon proto-oncogènes :")
display(onco)
print("Échantillon suppresseurs de tumeur :")
display(suppr)

Échantillon proto-oncogènes :


,accession,gene,length,label
0,P00519,ABL1,1130,oncogene
1,P04198,MYCN,464,oncogene
2,P04201,MAS1,325,oncogene
3,P08620,FGF4,206,oncogene
4,P08922,ROS1,2347,oncogene


Échantillon suppresseurs de tumeur :


,accession,gene,length,label
0,A0A2R8Y7D0,TINCR,87,tumor_suppressor
1,O43240,KLK10,276,tumor_suppressor
2,O60566,BUB1B,1050,tumor_suppressor
3,O95980,RECK,971,tumor_suppressor
4,P20827,EFNA1,205,tumor_suppressor


## 2. AlphaFold — accession → structure 3D

On interroge `…/api/prediction/{accession}`. L'API renvoie un JSON contenant l'URL
du fichier `.cif` **courant** (champ `cifUrl`). On télécharge ce fichier : c'est notre
donnée brute de structure pour la zone *raw*.

Le `try/except` est volontaire : **toutes les protéines n'ont pas de structure
AlphaFold**. La robustesse (loguer et écarter proprement) fait partie du barème, donc
on traite ce cas dès la faisabilité.

In [8]:
def fetch_alphafold_cif(accession: str, out_dir: str = ".") -> str | None:
    '''Télécharge le .cif AlphaFold courant. Retourne le chemin, ou None si absent.'''
    try:
        meta = session.get(f"{AFDB_API}/{accession}", timeout=30)
        meta.raise_for_status()
        entries = meta.json()
        if not entries:
            print(f"[skip] {accession} : aucune prédiction AlphaFold")
            return None

        cif_url = entries[0].get("cifUrl")
        if not cif_url:
            print(f"[skip] {accession} : pas de cifUrl dans la réponse API")
            return None

        cif = session.get(cif_url, timeout=60)
        cif.raise_for_status()
        path = f"{out_dir}/AF-{accession}.cif"
        with open(path, "wb") as f:
            f.write(cif.content)
        print(f"[ok]   {accession} : structure téléchargée ({len(cif.content)//1024} Ko)")
        return path
    except requests.HTTPError as e:
        print(f"[err]  {accession} : {e}")
        return None

cif_paths = {}
for acc in TARGETS:
    cif_paths[acc] = fetch_alphafold_cif(acc, out_dir="..")
    time.sleep(0.2)

[ok]   P04637 : structure téléchargée (354 Ko)
[ok]   P01116 : structure téléchargée (182 Ko)
[ok]   P06400 : structure téléchargée (840 Ko)


## 3. Extraction de features

C'est le cœur de la zone *staging*. Depuis le `.cif` + la séquence UniProt, on calcule :

- **pLDDT moyen** — confiance moyenne d'AlphaFold (stockée dans le champ B-factor des atomes)
- **% de résidus faible confiance** (pLDDT < 70) — *proxy de désordre*, l'intuition
  biologique centrale du projet (les protéines du cancer ont souvent beaucoup de
  régions désordonnées)
- **longueur de séquence**
- **rayon de giration** — compacité de la structure (sur les carbones α)
- **composition en acides aminés** — 20 fractions (calculées sur la séquence)

> Le *nombre de domaines* (listé dans le README) nécessite un clustering de la matrice
> PAE ; on le garde pour l'étape staging complète, pas pour la faisabilité.

In [9]:
AA20 = "ACDEFGHIKLMNPQRSTVWY"

def amino_acid_composition(sequence: str) -> dict:
    '''Fraction de chaque acide aminé standard dans la séquence.'''
    n = len(sequence) if sequence else 0
    counts = {f"aa_{a}": (sequence.count(a) / n if n else 0.0) for a in AA20}
    return counts

def extract_structure_features(cif_path: str) -> dict:
    '''pLDDT et rayon de giration à partir du .cif (carbones α).'''
    structure = gemmi.read_structure(cif_path)
    model = structure[0]

    plddt = []        # B-factor des CA = pLDDT par résidu
    ca_coords = []    # coordonnées des CA pour le rayon de giration
    for chain in model:
        for residue in chain:
            ca = residue.find_atom("CA", "*")
            if ca is not None:
                plddt.append(ca.b_iso)
                ca_coords.append([ca.pos.x, ca.pos.y, ca.pos.z])

    plddt = np.array(plddt)
    coords = np.array(ca_coords)

    # Rayon de giration : sqrt(moyenne des distances² au centre de masse)
    center = coords.mean(axis=0)
    rg = float(np.sqrt(((coords - center) ** 2).sum(axis=1).mean()))

    return {
        "n_residues_structure": int(len(plddt)),
        "plddt_mean": float(plddt.mean()),
        "pct_low_confidence": float((plddt < 70).mean() * 100),  # proxy désordre
        "radius_of_gyration": rg,
    }

# Test sur TP53
demo_feats = extract_structure_features(cif_paths["P04637"])
demo_feats

{'n_residues_structure': 393,
 'plddt_mean': 75.05012732547051,
 'pct_low_confidence': 40.20356234096692,
 'radius_of_gyration': 33.60058226273135}

## 4. Table ML-ready

On assemble tout : un vecteur de features par protéine + son label. C'est exactement
la forme qu'aura la zone *curated* qui alimente le Random Forest.

In [10]:
rows = []
for acc, (gene, label) in TARGETS.items():
    entry = fetch_uniprot_entry(acc)
    if cif_paths.get(acc) is None:
        print(f"[skip] {gene} : pas de structure, écarté du jeu ML")
        continue

    feats = extract_structure_features(cif_paths[acc])
    feats.update(amino_acid_composition(entry["sequence"]))
    feats.update({
        "accession": acc,
        "gene": gene,
        "seq_length": entry["length"],
        "label": label,
    })
    rows.append(feats)
    time.sleep(0.2)

df = pd.DataFrame(rows).set_index("gene")
# colonnes lisibles en premier
front = ["accession", "label", "seq_length", "n_residues_structure",
         "plddt_mean", "pct_low_confidence", "radius_of_gyration"]
df = df[front + [c for c in df.columns if c not in front]]
df

,accession,label,seq_length,n_residues_structure,plddt_mean,pct_low_confidence,radius_of_gyration,aa_A,aa_C,aa_D,...,aa_M,aa_N,aa_P,aa_Q,aa_R,aa_S,aa_T,aa_V,aa_W,aa_Y
gene,,,,,,,,,,,,,,,,,,,,,
TP53,P04637,tumor_suppressor,393,393,75.050127,40.203562,33.600582,0.061069,0.025445,0.050891,...,0.030534,0.035623,0.114504,0.038168,0.066158,0.096692,0.055980,0.045802,0.010178,0.022901
KRAS,P01116,oncogene,189,189,91.523862,7.407407,19.033957,0.047619,0.026455,0.074074,...,0.026455,0.021164,0.026455,0.052910,0.063492,0.047619,0.068783,0.084656,0.000000,0.047619
RB1,P06400,tumor_suppressor,928,928,76.066315,35.775862,36.464105,0.047414,0.016164,0.048491,...,0.030172,0.048491,0.064655,0.037716,0.049569,0.085129,0.059267,0.049569,0.007543,0.030172


## 5. Conclusion

Si cette table s'affiche avec une ligne par protéine et des features non vides, la
faisabilité est **validée** :

1. ✅ UniProt traduit gène → accession + séquence + label (par accession **et** par mot-clé)
2. ✅ AlphaFold renvoie une structure 3D exploitable via son API (robuste aux versions)
3. ✅ On extrait des features numériques propres, prêtes pour un classifieur

**Différence cancer vs forme honnête :** avec seulement 3 protéines on ne *teste* rien
statistiquement — le but ici est uniquement de prouver que la *plomberie* fonctionne.
La vraie question scientifique (peut-on séparer oncogène de suppresseur ?) viendra avec
le dataset complet (~560 + ~631 protéines) et le Random Forest.

### Étape suivante (étape 2)
Squelette du repo : structure de dossiers, `docker-compose` (LocalStack), `dvc.yaml`,
et migration de ces fonctions de faisabilité vers `src/ingest/` et `src/features/`.